# Análise exploratória | ZT502PO | tbl_ds_pro_zt502po_estrategia_liberacao_pedido

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`` |
| Ambiente | DEV |
| Objeto | Tabela |
| Colunas do DESCRIBE | 12 |
| Tipos | date, int, string |
| Clustering declarado | Não informado no CSV |
| Centro | Não consta; `cod_componente_versao` é apenas recorte alternativo, NÃO centro |
| Chave candidata | `num_documento` + `cod_componente_versao` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_ZT502PO.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | Sem cod_centro: não presumir equivalência do filtro SAP por centro. |
| 🔴 ALTA | 2 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;


### 1.1. Detalhes e histórico da tabela

Comandos independentes, executados somente em tabelas.


In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;


In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `num_documento`) AS distintos_num_documento, COUNT(DISTINCT `cod_componente_versao`) AS cod_componente_versao_distintos_nao_centros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;


## 3. Distribuição por centro

O schema não contém cod_centro; a consulta abaixo mostra somente um recorte alternativo, sem equivalência ao centro SAP.


In [0]:
SELECT `cod_componente_versao` AS recorte_alternativo_nao_centro, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_componente_versao` ORDER BY linhas DESC LIMIT 100;


## 3.1. Centros 4128 e 4014

Não existe campo de centro confirmado. Esta célula registra a limitação sem aplicar filtro inventado.


In [0]:
SELECT 'SEM_COLUNA_CENTRO' AS situacao, '4128 e 4014 nao podem ser medidos diretamente neste schema' AS observacao;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_documento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) k UNION ALL SELECT 'num_documento + cod_componente_versao' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `cod_componente_versao` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) k UNION ALL SELECT 'num_documento + cod_componente_versao + cod_liberacao' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `cod_componente_versao`, `cod_liberacao` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_documento`, t.`cod_componente_versao`, md5(to_json(struct(t.`num_documento`, t.`cod_componente_versao`, t.`cod_liberacao`, t.`cod_motivo`, t.`nm_responsavel_pela_modificacao_objeto`, t.`nm_usuario`, t.`dt_ultima_modificacao`, t.`dh_modificacao`, t.`qt_contagem`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` t), d AS (SELECT `num_documento`, `cod_componente_versao`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento`, `cod_componente_versao` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_documento`, `cod_componente_versao`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`num_documento` AS STRING)) AS `dist_num_documento`, COUNT(DISTINCT CAST(`cod_componente_versao` AS STRING)) AS `dist_cod_componente_versao`, COUNT(DISTINCT CAST(`cod_liberacao` AS STRING)) AS `dist_cod_liberacao`, COUNT(DISTINCT CAST(`cod_motivo` AS STRING)) AS `dist_cod_motivo`, COUNT(DISTINCT CAST(`dt_ultima_modificacao` AS STRING)) AS `dist_dt_ultima_modificacao`, COUNT(DISTINCT CAST(`dh_modificacao` AS STRING)) AS `dist_dh_modificacao`, COUNT(DISTINCT CAST(`qt_contagem` AS STRING)) AS `dist_qt_contagem`, COUNT(DISTINCT CAST(`dateingest` AS STRING)) AS `dist_dateingest`, COUNT(DISTINCT CAST(`yearingest` AS STRING)) AS `dist_yearingest`, COUNT(DISTINCT CAST(`monthingest` AS STRING)) AS `dist_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `num_documento`, `cod_componente_versao` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_num_documento` > 1) AS `chaves_variam_num_documento`, COUNT_IF(`dist_cod_componente_versao` > 1) AS `chaves_variam_cod_componente_versao`, COUNT_IF(`dist_cod_liberacao` > 1) AS `chaves_variam_cod_liberacao`, COUNT_IF(`dist_cod_motivo` > 1) AS `chaves_variam_cod_motivo`, COUNT_IF(`dist_dt_ultima_modificacao` > 1) AS `chaves_variam_dt_ultima_modificacao`, COUNT_IF(`dist_dh_modificacao` > 1) AS `chaves_variam_dh_modificacao`, COUNT_IF(`dist_qt_contagem` > 1) AS `chaves_variam_qt_contagem`, COUNT_IF(`dist_dateingest` > 1) AS `chaves_variam_dateingest`, COUNT_IF(`dist_yearingest` > 1) AS `chaves_variam_yearingest`, COUNT_IF(`dist_monthingest` > 1) AS `chaves_variam_monthingest` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(12, 'num_documento', 'string', c0_n, c0_v, c0_z, 'cod_componente_versao', 'string', c1_n, c1_v, c1_z, 'cod_liberacao', 'string', c2_n, c2_v, c2_z, 'cod_motivo', 'string', c3_n, c3_v, c3_z, 'nm_responsavel_pela_modificacao_objeto', 'string', c4_n, c4_v, c4_z, 'nm_usuario', 'string', c5_n, c5_v, c5_z, 'dt_ultima_modificacao', 'date', c6_n, c6_v, c6_z, 'dh_modificacao', 'string', c7_n, c7_v, c7_z, 'qt_contagem', 'int', c8_n, c8_v, c8_z, 'dateingest', 'date', c9_n, c9_v, c9_z, 'yearingest', 'string', c10_n, c10_v, c10_z, 'monthingest', 'string', c11_n, c11_v, c11_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_documento` IS NULL) AS c0_n, COUNT_IF(`num_documento` IS NOT NULL AND LOWER(TRIM(`num_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`cod_componente_versao` IS NULL) AS c1_n, COUNT_IF(`cod_componente_versao` IS NOT NULL AND LOWER(TRIM(`cod_componente_versao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`cod_componente_versao`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`cod_liberacao` IS NULL) AS c2_n, COUNT_IF(`cod_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`cod_motivo` IS NULL) AS c3_n, COUNT_IF(`cod_motivo` IS NOT NULL AND LOWER(TRIM(`cod_motivo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`cod_motivo`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NULL) AS c4_n, COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NOT NULL AND LOWER(TRIM(`nm_responsavel_pela_modificacao_objeto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`nm_responsavel_pela_modificacao_objeto`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`nm_usuario` IS NULL) AS c5_n, COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`dt_ultima_modificacao` IS NULL) AS c6_n, CAST(0 AS BIGINT) AS c6_v, CAST(0 AS BIGINT) AS c6_z, COUNT_IF(`dh_modificacao` IS NULL) AS c7_n, COUNT_IF(`dh_modificacao` IS NOT NULL AND LOWER(TRIM(`dh_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c7_v, COUNT_IF(TRIM(`dh_modificacao`) RLIKE '^0+([.,]0+)?$') AS c7_z, COUNT_IF(`qt_contagem` IS NULL) AS c8_n, CAST(0 AS BIGINT) AS c8_v, COUNT_IF(`qt_contagem` = 0) AS c8_z, COUNT_IF(`dateingest` IS NULL) AS c9_n, CAST(0 AS BIGINT) AS c9_v, CAST(0 AS BIGINT) AS c9_z, COUNT_IF(`yearingest` IS NULL) AS c10_n, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`monthingest` IS NULL) AS c11_n, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS c11_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte ou sem preenchimento declarado

Confere o comportamento documentado; 100% nulo não é automaticamente erro de carga.


In [0]:
WITH a AS (SELECT 'cod_motivo' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_motivo` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'nm_responsavel_pela_modificacao_objeto' AS coluna, COUNT(*) AS total, COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'qt_contagem' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_contagem` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) SELECT coluna, total, nulos, CASE WHEN total = nulos THEN 'CONFIRMADO NO RECORTE' ELSE 'HA PREENCHIMENTO: INVESTIGAR' END AS leitura FROM a ORDER BY coluna;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `num_documento`) AS d0, COUNT(DISTINCT `cod_componente_versao`) AS d1, COUNT(DISTINCT `cod_liberacao`) AS d2, COUNT(DISTINCT `cod_motivo`) AS d3 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(4, 'num_documento', d0, 'cod_componente_versao', d1, 'cod_liberacao', d2, 'cod_motivo', d3) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_componente_versao' AS coluna, CAST(`cod_componente_versao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_componente_versao` UNION ALL SELECT 'cod_liberacao' AS coluna, CAST(`cod_liberacao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_liberacao` UNION ALL SELECT 'cod_motivo' AS coluna, CAST(`cod_motivo` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_motivo`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

O schema não apresenta indicadores selecionáveis sem dados pessoais.


In [0]:
SELECT 'Sem indicadores nao pessoais no schema' AS observacao;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(1, 'qt_contagem', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`qt_contagem`) AS n0_min, MAX(`qt_contagem`) AS n0_max, AVG(`qt_contagem`) AS n0_avg, PERCENTILE_APPROX(`qt_contagem`, 0.5) AS n0_med, COUNT_IF(`qt_contagem` < 0) AS n0_neg, COUNT_IF(`qt_contagem` = 0) AS n0_zero FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(1, 'qt_contagem', CAST(s0 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`qt_contagem`) AS s0 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_ultima_modificacao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_ultima_modificacao` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_ultima_modificacao` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_ultima_modificacao`) AS STRING) AS menor_literal, CAST(MAX(`dt_ultima_modificacao`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'dh_modificacao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dh_modificacao` IS NULL) AS nulos, COUNT_IF(TRIM(`dh_modificacao`) = '') AS vazios, COUNT_IF(TRIM(`dh_modificacao`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dh_modificacao`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dh_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dh_modificacao`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dh_modificacao`) AS menor_literal, MAX(`dh_modificacao`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dateingest` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'num_documento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'cod_componente_versao' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_componente_versao`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_componente_versao`))) AS tamanho_max, COUNT_IF(TRIM(`cod_componente_versao`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_componente_versao`) RLIKE '^[0-9]+$' THEN TRIM(`cod_componente_versao`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_componente_versao`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_componente_versao`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'cod_liberacao' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_liberacao`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_liberacao`))) AS tamanho_max, COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_liberacao`) RLIKE '^[0-9]+$' THEN TRIM(`cod_liberacao`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_liberacao`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_liberacao`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` UNION ALL SELECT 'cod_motivo' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_motivo`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_motivo`))) AS tamanho_max, COUNT_IF(TRIM(`cod_motivo`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_motivo`) RLIKE '^[0-9]+$' THEN TRIM(`cod_motivo`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_motivo`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_motivo`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`num_documento`, t.`cod_componente_versao`, t.`cod_liberacao`, t.`cod_motivo`, t.`nm_responsavel_pela_modificacao_objeto`, t.`nm_usuario`, t.`dt_ultima_modificacao`, t.`dh_modificacao`, t.`qt_contagem`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `num_documento`, `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, `dt_ultima_modificacao`, `dh_modificacao`, `qt_contagem`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_responsavel_pela_modificacao_objeto` AS STRING)),1,8) AS `nm_responsavel_pela_modificacao_objeto_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_componente_versao`, `cod_liberacao`, `cod_motivo` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Mede datas de carga e partições; não confundir carimbo de ingestão com evento de negócio.


In [0]:
SELECT COUNT(*) AS linhas, MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, COUNT(DISTINCT `dateingest`) AS `distintos_dateingest`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, COUNT(DISTINCT `yearingest`) AS `distintos_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest`, COUNT(DISTINCT `monthingest`) AS `distintos_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Recorte PO, status e lacunas documentadas


In [0]:
SELECT `cod_componente_versao`, COUNT(*) AS linhas, COUNT(DISTINCT `num_documento`) AS pedidos_distintos, COUNT_IF(`cod_motivo` IS NOT NULL) AS motivo_preenchido, COUNT_IF(`qt_contagem` IS NOT NULL) AS contagem_preenchida, COUNT_IF(`dt_ultima_modificacao` IS NULL) AS data_modificacao_nula FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `cod_componente_versao` ORDER BY linhas DESC;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `num_documento`, `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, `dt_ultima_modificacao`, `dh_modificacao`, `qt_contagem`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_responsavel_pela_modificacao_objeto` AS STRING)),1,8) AS `nm_responsavel_pela_modificacao_objeto_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` ;

## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_documento`, `cod_componente_versao`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` GROUP BY `num_documento`, `cod_componente_versao`), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`) SELECT 'ZT502PO' AS transacao, 'tbl_ds_pro_zt502po_estrategia_liberacao_pedido' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, 'SEM CENTRO NO SCHEMA' AS filtro_centro, 'Chave candidata: num_documento + cod_componente_versao' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
